In [1]:
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)

In [2]:
a = np.arange(10)
b = jnp.array(a, dtype='float64')

In [3]:
def f(x, y):
    return x**2 + y

In [6]:
grad_fn = jax.vmap(jax.grad(f, argnums=[0, 1]))

In [7]:
grad_fn(b,b)

(Array([ 0.,  2.,  4.,  6.,  8., 10., 12., 14., 16., 18.], dtype=float64),
 Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float64))

In [8]:
gradf = jax.jacrev(f)

In [8]:
print(jax.devices("cpu"))
print(jax.devices("gpu"))

[CpuDevice(id=0)]
[CudaDevice(id=0)]


In [2]:
from old.tests.models import hes1

In [3]:
hes1

<module 'old.tests.models.hes1' from '/home/jamie/storage-1/github-repos/magi_msvgd/magi_msvgd/old/tests/models/hes1.py'>

In [16]:
jnp.log(b)

Array([      -inf, 0.        , 0.69314718, 1.09861229, 1.38629436,
       1.60943791, 1.79175947, 1.94591015, 2.07944154, 2.19722458],      dtype=float64)

In [13]:
jnp.stack([b,b])

Array([[0., 1., 2., 3., 4., 5., 6., 7., 8., 9.],
       [0., 1., 2., 3., 4., 5., 6., 7., 8., 9.]], dtype=float64)

In [101]:
import numpy as np
import jax
import jax.numpy as jnp

tau = [jnp.linspace(0, 240, int(240/15 +1)),
       jnp.linspace(7.5, 232.5, int((232.5-7.5)/15 +1)),
       jnp.array([])]

hyperparameters = {
    "theta" : jnp.array([0.022, 0.3, 0.031, 0.028, 0.5, 20, 0.3]),
    "X0" : jnp.log(jnp.array([1.438575, 2.037488, 17.90385])),
    "sigma" : jnp.array([0.15, 0.15, np.nan]),
    "tau" : tau,
    # "obs_times" : test_helpers.obs_matrix(tau),
    "I" : jnp.linspace(0, 240, int(240/7.5 +1))
}

def ode(X, theta, t=None):
    '''
    X : n x D
    theta : p

    return : n x D
    '''
    # n = X.shape[0]
    P, M, H = jnp.exp(X.T)
    a, b, c, d, e, f, g = theta

    return jnp.stack([-a*H + b*M/P - c,
                        -d + e/(1+P**2)/M,
                       -a*P + f/(1+P**2)/H - g])


def dfdx(X, theta, t=None):
    '''
    X : n x D
    theta : p

    return : n x D x D
    '''
    n = X.shape[0]
    logP, logM, logH = X.T
    a, b, c, d, e, f, g = jnp.tile(theta, [n, 1]).T
    
    zero = jnp.zeros(n)
    dP = -(1 + jnp.exp(2*logP))**-2 * jnp.exp(2*logP) * 2

    return jnp.stack([jnp.stack([-b*jnp.exp(logM - logP), b*jnp.exp(logM - logP), -a*jnp.exp(logH)], axis=1),
                        jnp.stack([e*jnp.exp(-logM)*dP, -e*jnp.exp(-logM)/(1+jnp.exp(2*logP)), zero], axis=1),
                        jnp.stack([-a*jnp.exp(logP) + f*jnp.exp(-logH)*dP, zero, -f*jnp.exp(-logH)/(1+jnp.exp(2*logP))], axis=1)], axis=2)

X = jnp.array(np.random.normal(size=[30, 3]))
theta = jnp.array(np.random.normal(size=7))

In [105]:
tau = [jnp.linspace(0, 20, 41),
       jnp.linspace(0, 20, 41)]

hyperparameters = {
    "theta" : jnp.array([0.2, 0.2, 3.0]),
    "X0" : jnp.array([-1.0, 1.0]),
    "sigma" : jnp.array([0.2, 0.2]),
    "tau" : tau,
    # "obs_times" : test_helpers.obs_matrix(tau),
    "I" : jnp.linspace(0, 20, int(160 +1))
}

def ode(X, theta, t=None):
    '''
    X : n x D
    theta : p

    return : n x D
    '''
    n = X.shape[0]
    V, R = X.T
    a, b, c = theta # theta.repeat([n, 1]).T

    return jnp.stack([c * (V - V**3/3 + R),
                        -1/c * (V - a + b*R)])


def dfdx(X, theta, t=None):
    '''
    X : n x D
    theta : p

    return : n x D x D
    '''
    n = X.shape[0]
    V, R = X.T
    a, b, c = jnp.tile(theta, [n, 1]).T

    return jnp.stack([jnp.stack([c * (1 - V**2), c], axis=1),
                        jnp.stack([-1/c, -b/c], axis=1)], axis=2)

def dfdtheta(X, theta, t=None):
    '''
    X : n x D
    theta : p

    return : n x p x D
    '''
    n = X.shape[0]
    V, R = X.T
    a, b, c = jnp.tile(theta, [n, 1]).T
    
    zero = jnp.zeros(n)
    
    return jnp.stack([jnp.stack([zero, zero, V - V**3/3 + R], axis=1),
                        jnp.stack([1/c, -R/c, (V - a + b*R)/c**2], axis=1)], axis=2)
    
X = jnp.array(np.random.normal(size=[30, 2]))
theta = jnp.array(np.random.normal(size=3))

In [119]:
jac_fn = jax.vmap(jax.jacobian(ode, argnums=0), in_axes=(0, None, None))
jnp.permute_dims(jac_fn(X, theta, None), [0, 2, 1])

Array([[[-4.49964220e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.38945974e-01,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-4.87419980e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 4.15999643e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-2.46559755e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-4.73317115e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.30678271e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.64986381e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.97658073e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.56996562e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.79780059e-01,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-

In [120]:
dfdx(X, theta)

Array([[[-4.49964220e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.38945974e-01,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-4.87419980e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 4.15999643e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-2.46559755e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-4.73317115e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.30678271e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.64986381e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.97658073e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-3.56996562e-02,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[ 1.79780059e-01,  2.04973378e+01],
        [-4.87868233e-02,  2.14999597e+01]],

       [[-

In [180]:
foo = jnp.array([1,2,3])

In [178]:
def jnp_pad(array):
    return jnp.expand_dims(array, axis=-1)

In [181]:
jnp.tile(foo, [20, 1])

Array([[1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3],
       [1, 2, 3]], dtype=int64)

In [190]:
jnp.tile(foo.reshape(1, -1), [20,1]).shape

(20, 3)

In [191]:
jnp.abs(b)

Array([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.], dtype=float64)

In [193]:
np.random.normal(loc=X, scale=0.2)

array([[-0.52309149, -0.12040695],
       [-2.04610042,  0.46016587],
       [-0.14387738,  0.33681226],
       [ 1.34733839,  0.07584215],
       [ 0.8795542 , -0.81589628],
       [-0.36890082, -1.51589619],
       [ 0.61637361, -0.19936031],
       [ 0.81081745,  0.47236478],
       [-0.73000574, -1.60819509],
       [ 0.77376152, -2.72039525],
       [-2.13717475,  0.90682379],
       [-0.17045397, -0.14283738],
       [ 1.82951487,  2.20974333],
       [ 0.83973199,  0.38766937],
       [ 1.15505191,  0.19328532],
       [-1.205672  , -0.32468259],
       [-0.0355088 , -1.59586391],
       [ 0.26529214, -1.2092749 ],
       [ 0.42769518, -0.20325696],
       [-1.63135642,  1.56637744],
       [-0.01456993,  0.15841489],
       [ 1.55369507,  0.18328031],
       [-0.46918569, -0.29417199],
       [ 0.16470438,  1.31918833],
       [ 0.14918131, -1.57164052],
       [-1.46300359, -1.32096508],
       [ 1.23101944,  0.05892803],
       [ 0.09321071,  0.9972335 ],
       [-0.97774804,

In [270]:
@jax.jit
def square_distances(tensor):
    # faster than torch.cdist
    diffs = jnp.tile(tensor, [tensor.shape[0], 1, 1]) - jnp.expand_dims(tensor, 1)
    return jnp.linalg.norm(diffs, axis=-1)

In [284]:
@jax.jit
def square_distances(A):
    return jnp.sum((A[:, jnp.newaxis, :] - A[jnp.newaxis, :, :])**2, axis=-1)

In [235]:
def pairwise_sq_dist_self(X):
    X_sq = jnp.sum(X ** 2, axis=-1, keepdims=True)  # n x 1
    cross = X @ X.T                                  # n x n
    return X_sq + X_sq.T - 2 * cross

In [229]:
A = jnp.array(np.random.normal(500, 400))

In [288]:
jnp.allclose(square_distances(X), cdist(X,X)**2)

Array(True, dtype=bool)

In [291]:
import jax.scipy as jsp

In [293]:
jsp.special.gamma

<function jax._src.scipy.special.gamma(x: 'ArrayLike') -> 'Array'>

In [296]:
jsp.special.kve

AttributeError: module 'jax.scipy.special' has no attribute 'kve'

In [231]:
from scipy.spatial.distance import cdist

In [232]:
%timeit jnp.array(cdist(X, X)**2)

122 μs ± 2.36 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [287]:
%timeit square_distances(X)

27.7 μs ± 2.48 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [236]:
%timeit pairwise_sq_dist_self(X)

389 μs ± 2.32 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [196]:
jnp.diag_p

AttributeError: module 'jax.numpy.linalg' has no attribute 'diag'

In [150]:
jnp.tile(jnp.array([0,1,2], 10, dtype='float32', device=jax.devices('cpu')[0]))

TypeError: array() got multiple values for argument 'dtype'

In [ ]:
jnp.repeat

In [146]:
jnp.tile(jnp.arange(30).reshape(-1, 1), [10])

Array([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
       [ 2,  2,  2,  2,  2,  2,  2,  2,  2,  2],
       [ 3,  3,  3,  3,  3,  3,  3,  3,  3,  3],
       [ 4,  4,  4,  4,  4,  4,  4,  4,  4,  4],
       [ 5,  5,  5,  5,  5,  5,  5,  5,  5,  5],
       [ 6,  6,  6,  6,  6,  6,  6,  6,  6,  6],
       [ 7,  7,  7,  7,  7,  7,  7,  7,  7,  7],
       [ 8,  8,  8,  8,  8,  8,  8,  8,  8,  8],
       [ 9,  9,  9,  9,  9,  9,  9,  9,  9,  9],
       [10, 10, 10, 10, 10, 10, 10, 10, 10, 10],
       [11, 11, 11, 11, 11, 11, 11, 11, 11, 11],
       [12, 12, 12, 12, 12, 12, 12, 12, 12, 12],
       [13, 13, 13, 13, 13, 13, 13, 13, 13, 13],
       [14, 14, 14, 14, 14, 14, 14, 14, 14, 14],
       [15, 15, 15, 15, 15, 15, 15, 15, 15, 15],
       [16, 16, 16, 16, 16, 16, 16, 16, 16, 16],
       [17, 17, 17, 17, 17, 17, 17, 17, 17, 17],
       [18, 18, 18, 18, 18, 18, 18, 18, 18, 18],
       [19, 19, 19, 19, 19, 19, 19, 19, 19, 19],
       [20, 20, 20, 

In [138]:
jnp.tile(jnp.expand_dims(b, axis=0), 3)

Array([[0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 0., 1., 2., 3., 4., 5.,
        6., 7., 8., 9., 0., 1., 2., 3., 4., 5., 6., 7., 8., 9.]],      dtype=float64)

In [134]:
jnp.expand_dims(b, axis=0)

Array([[0., 1., 2., 3., 4., 5., 6., 7., 8., 9.]], dtype=float64)

In [133]:
jnp.tile(b, [1,2,3])

Array([[[0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 0., 1., 2., 3., 4., 5.,
         6., 7., 8., 9., 0., 1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 0., 1., 2., 3., 4., 5.,
         6., 7., 8., 9., 0., 1., 2., 3., 4., 5., 6., 7., 8., 9.]]],      dtype=float64)

In [131]:
jnp.expand_dims(b, axis=-1)

Array([[0.],
       [1.],
       [2.],
       [3.],
       [4.],
       [5.],
       [6.],
       [7.],
       [8.],
       [9.]], dtype=float64)

In [121]:
jac_fn = jax.vmap(jax.jacobian(ode, argnums=1), in_axes=(0, None, None))
jnp.permute_dims(jac_fn(X, theta, None), [0, 2, 1]).shape

(30, 3, 2)

In [116]:
dfdtheta(X, theta)

Array([[[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00, -6.12685664e+00],
        [-5.70426163e-01, -5.60620775e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00,  8.73618464e+00],
        [ 9.80715054e-01, -9.48122231e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00,  8.17253953e+00],
        [ 3.68409814e-01, -1.48811608e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00,  7.28280945e+00],
        [ 8.75853746e-01,  4.16662852e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00, -1.77091044e+01],
        [-2.76633495e-01, -3.97049463e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00, -3.03881944e+01],
        [-1.65352829e+00, -1.03768974e+03]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.00000000e+00, -3.90199036e+00],
        [ 3.16296376e-01, -1.57196079e+02]],

       [[ 0.00000000e+00, -2.04973378e+01],
        [ 0.000000

In [57]:
ode(X, theta)

Array([[ 1.73288244e+00, -1.50691727e+00,  4.55449233e+00],
       [ 4.24293195e-01, -5.07095945e-01,  5.29706321e+00],
       [-1.77544477e+00, -7.54610816e-02,  5.56396924e+00],
       [-2.70334372e+00, -7.96562250e-01,  5.51168532e+00],
       [-4.59784138e+00,  9.87108724e-02,  5.62851225e+00],
       [ 4.56057601e+00,  5.45080386e-03,  6.03323951e+00],
       [-1.21637565e+00, -1.11320406e+00,  4.68069639e+00],
       [-1.96203083e+00,  1.07781416e-01,  7.17414936e+00],
       [ 5.99629915e-01, -5.02355533e-01,  3.78911279e+00],
       [-3.05575280e+00, -9.69811405e-01,  3.43351277e+00],
       [-2.41516453e+00,  9.69728147e-02,  6.22389846e+00],
       [ 1.57959480e+00, -2.15929879e+00,  3.69304752e+00],
       [-2.43621485e+00,  4.95503657e-02,  4.81117651e+00],
       [-1.34219397e-01, -1.62228110e-01,  5.42415136e+00],
       [-3.12106547e+00, -1.56296595e-01,  5.14693716e+00],
       [-1.69901540e+00, -7.26878916e-01,  5.02710010e+00],
       [ 2.80194737e+00, -2.66736189e+00

In [ ]:
J(X, theta))

In [60]:
jax.jacobian(ode)(X, theta).shape

(30, 3, 30, 3)